# Debug Hydromancer snapshot HTTP 403 responses

This notebook diagnoses the two REST snapshot calls required by the L2/L4 ingestion bootstrap:

- `l2BookDiffSnapshot`
- `l4Book`

It uses the documented `POST https://api.hydromancer.xyz/info` request with `Authorization: Bearer ...`. It never prints the API key, never places it in a URL, and limits error-body output. A 403 from both calls usually means that the key lacks one or both snapshot permissions, but the response body and safe response headers may provide a more specific diagnosis.

> Run this notebook from a trusted local environment. Do not commit `.env` or paste notebook output containing credentials.

In [ ]:
from __future__ import annotations

import json
import os
import urllib.error
import urllib.request
from pathlib import Path
from typing import Any

import msgpack
from dotenv import load_dotenv

INFO_URL = "https://api.hydromancer.xyz/info"


def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Run the notebook from inside the hydromancer-eda repository")


ROOT = find_repo_root()
# Re-read edited .env values whenever this setup cell is executed.
load_dotenv(ROOT / ".env", override=True)
API_KEY = os.environ.get("HYDROMANCER_API_KEY", "")
COINS = [coin.strip() for coin in os.environ.get("HYDROMANCER_COINS", "").split(",") if coin.strip()]

if not API_KEY:
    raise RuntimeError("HYDROMANCER_API_KEY is not configured")
if not COINS:
    raise RuntimeError("HYDROMANCER_COINS is not configured")

print({"credentials_loaded": True, "coins": COINS})  # Deliberately excludes the key.

## Safe request helper

Only an allowlist of non-secret response headers is retained. Error text is capped and any accidental occurrence of the API key is redacted before display. The request URL does not contain the token.

In [ ]:
SAFE_RESPONSE_HEADERS = {
    "content-type",
    "content-encoding",
    "x-payload-format",
    "x-compression",
    "x-request-id",
    "cf-ray",
    "retry-after",
}


def safe_headers(headers: Any) -> dict[str, str]:
    return {
        key.lower(): value
        for key, value in headers.items()
        if key.lower() in SAFE_RESPONSE_HEADERS
    }


def safe_error_text(body: bytes) -> str:
    text = body[:2_000].decode("utf-8", errors="replace")
    return text.replace(API_KEY, "[REDACTED]") if API_KEY else text


def post_info(
    payload: dict[str, Any],
    timeout: float = 30.0,
    user_agent: str | None = None,
) -> dict[str, Any]:
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
        "Accept": "application/octet-stream",
    }
    if user_agent is not None:
        headers["User-Agent"] = user_agent
    request = urllib.request.Request(
        INFO_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers=headers,
        method="POST",
    )
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return {
                "request_type": payload["type"],
                "status": response.status,
                "headers": safe_headers(response.headers),
                "body": response.read(),
                "error": None,
            }
    except urllib.error.HTTPError as exc:
        body = exc.read()
        return {
            "request_type": payload["type"],
            "status": exc.code,
            "headers": safe_headers(exc.headers),
            "body": body,
            "error": safe_error_text(body),
        }
    except urllib.error.URLError as exc:
        return {
            "request_type": payload["type"],
            "status": None,
            "headers": {},
            "body": b"",
            "error": f"Network error: {type(exc.reason).__name__}",
        }


def display_result(result: dict[str, Any]) -> None:
    print(
        json.dumps(
            {
                "request_type": result["request_type"],
                "status": result["status"],
                "headers": result["headers"],
                "error": result["error"],
                "response_bytes": len(result["body"]),
            },
            indent=2,
        )
    )

## Reproduce both requests

The payloads match the ingestion service. Coin names remain case-sensitive.

In [ ]:
REQUESTS = {
    "l2BookDiffSnapshot": {"type": "l2BookDiffSnapshot", "coins": COINS},
    "l4Book": {"type": "l4Book", "coins": COINS},
}

results = {name: post_info(payload) for name, payload in REQUESTS.items()}
for result in results.values():
    display_result(result)

## Interpret the baseline status codes

- **200:** Permission and payload are accepted; continue to the parser cells.
- **400:** Authentication reached the endpoint, but inspect coin spelling and payload shape.
- **403 with JSON:** May indicate a missing endpoint entitlement or invalid API key.
- **403 with Cloudflare HTML / error 1010:** The request was rejected by Cloudflare before normal API authorization, commonly because of the default `Python-urllib` client fingerprint. Test an explicit application `User-Agent` below.
- **404:** No snapshot is currently available for the requested market.
- **429:** Snapshot rate limit exceeded; honor `retry-after` and avoid repeated execution.
- **No status:** DNS, TLS, proxy, or other transport failure.

In [ ]:
statuses = {name: result["status"] for name, result in results.items()}
cloudflare_denial = any(
    result["status"] == 403
    and "text/html" in result["headers"].get("content-type", "")
    and "Cloudflare" in (result["error"] or "")
    for result in results.values()
)
if cloudflare_denial:
    conclusion = "Cloudflare rejected the baseline HTTP client before API authorization."
elif all(status == 403 for status in statuses.values()):
    conclusion = "Both endpoints returned an API-level 403; verify key entitlements."
elif all(status == 200 for status in statuses.values()):
    conclusion = "Both snapshot permissions are working. Validate decoding below."
else:
    conclusion = "The endpoints returned mixed results; inspect the safe diagnostics above."
print({"statuses": statuses, "cloudflare_denial": cloudflare_denial, "conclusion": conclusion})

## Retry with an explicit application User-Agent

Python's `urllib` otherwise sends its default client identity. Cloudflare error 1010 means that identity was blocked before the request reached Hydromancer's normal authorization path. This retry uses a stable, non-secret application identifier; it does not weaken TLS or bypass authentication.

In [ ]:
APPLICATION_USER_AGENT = "hydromancer-eda-debug/0.1"
user_agent_results = {
    name: post_info(payload, user_agent=APPLICATION_USER_AGENT)
    for name, payload in REQUESTS.items()
}
for result in user_agent_results.values():
    display_result(result)

effective_results = {
    name: user_agent_results[name]
    if user_agent_results[name]["status"] == 200
    else results[name]
    for name in REQUESTS
}
print({"effective_statuses": {name: result["status"] for name, result in effective_results.items()}})

## Decode a successful L2 snapshot

The documented msgpack snapshot is positional: `[coin, time, levels, height, epoch, seq]`. This cell prints only market-level metadata and level counts.

In [ ]:
def summarize_l2_snapshot(body: bytes) -> list[dict[str, Any]]:
    decoded = msgpack.unpackb(body, raw=False)
    snapshots = [decoded] if decoded and isinstance(decoded[0], str) else decoded
    summary = []
    for snapshot in snapshots:
        coin, time_ms, levels, height, epoch, seq = snapshot
        summary.append(
            {
                "coin": coin,
                "time_ms": time_ms,
                "height": height,
                "epoch": epoch,
                "seq": seq,
                "bid_levels": len(levels[0]),
                "ask_levels": len(levels[1]),
            }
        )
    return summary


l2_result = effective_results["l2BookDiffSnapshot"]
if l2_result["status"] == 200:
    print(json.dumps(summarize_l2_snapshot(l2_result["body"]), indent=2))
else:
    print("Skipped: l2BookDiffSnapshot did not return HTTP 200.")

## Decode a successful L4 snapshot

The documented multi-zstd envelope is `[count:u32][height:u64][timestamp_ms:u64][len:u32][blob]...`. Each decompressed msgpack market is positional as `[coin, bids, asks]`. Wallets and order details are deliberately not displayed.

In [ ]:
from hydromancer_eda.visualization.l4 import (
    animate_l4_book,
    l4_snapshot_overview,
    parse_l4_snapshot,
    stream_l4_book_updates,
)


def summarize_l4_snapshot(body: bytes) -> dict[str, Any]:
    """Decode the complete L4 snapshot, retaining orders and wallet attribution."""
    return parse_l4_snapshot(body)


l4_result = effective_results["l4Book"]
l4_snapshot = None
if l4_result["status"] == 200:
    l4_snapshot = summarize_l4_snapshot(l4_result["body"])
    # Print only counts; the complete wallet/order data remains in l4_snapshot.
    print(json.dumps(l4_snapshot_overview(l4_snapshot), indent=2))
else:
    print("Skipped: l4Book did not return HTTP 200.")

## Animate wallet ownership at each L4 price level

`animate_l4_book` accepts the object returned by `summarize_l4_snapshot`. Bids extend left, asks extend right, and every selected price level sums to 100%. Each color is a wallet; hover shows the full wallet, level size, and exact percentage.

`minimum_named_share=0` displays every wallet separately. Increase it (for example to `1.0`) to combine very small shares into `Other` when the legend becomes crowded.

In [ ]:
animator = None
if l4_snapshot is not None:
    animator = animate_l4_book(
        l4_snapshot,
        coin=COINS[0],
        level_count=12,
        minimum_named_share=0.0,
    )
else:
    print("Fetch a successful L4 snapshot before creating the animation.")

## Stream live L4 updates

Do not repeatedly poll `l4Book`: the REST snapshot is rate-limited. The coroutine below subscribes to `l4BookUpdates`, then fetches one fresh snapshot while WebSocket messages queue. It discards updates already covered by the snapshot height and applies each subsequent block in memory.

Rendering defaults to 5 frames per second to keep Jupyter responsive while every order update is still applied. Increase `max_fps` if your browser can keep up. Set `START_LIVE_STREAM = True` and run the cell. Cancel the returned task with the following cell.

In [ ]:
import asyncio


def fetch_fresh_l4_snapshot() -> dict[str, Any]:
    result = post_info(
        {"type": "l4Book", "coins": [COINS[0]]},
        user_agent=APPLICATION_USER_AGENT,
    )
    if result["status"] != 200:
        raise RuntimeError(f"l4Book snapshot failed with HTTP {result['status']}")
    return summarize_l4_snapshot(result["body"])


START_LIVE_STREAM = False
live_l4_task = None
if START_LIVE_STREAM:
    if animator is None:
        raise RuntimeError("Create the animator first")
    live_l4_task = asyncio.create_task(
        stream_l4_book_updates(
            animator,
            API_KEY,
            snapshot_fetcher=fetch_fresh_l4_snapshot,
            max_fps=5.0,
        )
    )
    print("Live L4 animation started.")
else:
    print("Set START_LIVE_STREAM = False to start live updates.")

In [ ]:
if live_l4_task is not None:
    live_l4_task.cancel()
    await asyncio.gather(live_l4_task, return_exceptions=True)
    print("Live L4 animation stopped.")

## Optional payload-shape check

A 403 is an authorization result, not evidence that the plural `coins` payload is malformed. If Hydromancer support asks you to compare payload forms, set `RUN_VARIANTS = True`. This makes two additional requests, so leave it disabled during normal debugging to respect snapshot rate limits.

In [ ]:
RUN_VARIANTS = False

if RUN_VARIANTS:
    variants = [
        {"type": "l2BookDiffSnapshot", "coin": COINS[0]},
        {"type": "l4Book", "coins": [COINS[0]]},
    ]
    for payload in variants:
        display_result(post_info(payload))
else:
    print("Optional additional requests are disabled.")

## Support checklist

If the explicit-User-Agent responses remain 403, provide Hydromancer support with:

1. The endpoint names `l2BookDiffSnapshot` and `l4Book`.
2. The HTTP status and error message shown above.
3. Any `x-request-id` or `cf-ray` response values.
4. The requested case-sensitive coin names.
5. Confirmation that WebSocket authentication succeeds.

Do **not** provide the API key, a token-bearing WebSocket URL, session ID, or cursor.